# R1-C01 — Diagnóstico independiente de factibilidad FINAL 2024 · V01

**Objetivo único:** auditar las 15 restricciones del escalado global \(\alpha\) sobre `POST_CANDIDATE` de 2024, identificar las restricciones activas y cuantificar la brecha. **No genera FINAL ni modifica RAW o POST.**

- Entrada: `R1_C01_2024_POST_FINAL_V01.zip` **inmutable**; climatología NSRDB 1998–2022 ya incorporada en las restricciones auditadas.
- Unidades: irradiación integrada en kWh/m². Intervalos de alpha adimensionales.
- Regla canónica: tres restricciones anuales de GHI/DNI/DHI y doce mensuales de GHI; fracciones difusas mensuales se preservan mediante un factor global común y **no añaden intervalos de alpha**.
- Toda sensibilidad numérica es **diagnóstica**, nunca constituye autorización para relajar el protocolo.
- Año bisiesto: verificar 366 días, incluyendo 48 registros de 2024-02-29.
- El input se identifica por SHA-256 de la ejecución recibida de Colab; puede tener sufijo `(1)` al descargarse, pero cargarlo en `/content/` con el nombre canónico sin sufijo.


In [ ]:
from pathlib import Path
import hashlib, zipfile, io, json, platform
import numpy as np
import pandas as pd

YEAR=2024
NAME='R1_C01_2024_POST_FINAL_V01.zip'
EXPECTED_ZIP_SHA='a8aa5c8cf9793588ad3bd97c758b726c84992150dcbc538b5fbca1dec1524476'
EXPECTED_CONSTRAINTS_SHA='caebe69bf93589ebc05f931bb339b0a73938048d03caacf8f40d00c9e9f0a218'
EXPECTED_RAW_ZIP_SHA='bca48774f606c309ca9a973681c926031c0bf1943518b7241304d39ab8cacbe8'
OUTPUT_NAME='R1_C01_2024_FINAL_Feasibility_Audit_V01'
# Solo carpeta NUEVA. Nunca se escribe en el ZIP de entrada ni sobre archivos POST/FINAL.
BASE=Path('/content') if Path('/content/'+NAME).is_file() else Path.cwd()
if not (BASE/NAME).is_file() and Path('/mnt/data/'+NAME).is_file(): BASE=Path('/mnt/data')
SOURCE=BASE/NAME
OUTPUT=BASE/OUTPUT_NAME
assert SOURCE.is_file(), f'No se encuentra {NAME} en {BASE}'
assert not OUTPUT.exists(), f'Carpeta de resultados ya existente: {OUTPUT}. Conservar ejecución previa; usar otra carpeta.'

def sha256_bytes(data): return hashlib.sha256(data).hexdigest()
def sha256_file(path): return sha256_bytes(Path(path).read_bytes())
assert sha256_file(SOURCE)==EXPECTED_ZIP_SHA, 'ZIP fuente no coincide con la evidencia aprobada'
print('PASS identidad ZIP fuente:', sha256_file(SOURCE))

In [ ]:
# Inspección inmutable de los productos aprobados
with zipfile.ZipFile(SOURCE, 'r') as z:
    assert z.testzip() is None, 'ZIP corrupto'
    names=set(z.namelist())
    required={
        'R1_C01_2024_POST_FINAL_15Constraints_V01.csv',
        'R1_C01_2024_POST_FINAL_Manifest_V01.json',
        'R1_C01_2024_POST_MonthlyAudit_V01.csv',
        'R1_C01_2024_POST_CANDIDATE_V01.csv'
    }
    assert required <= names, 'Faltan productos del POST auditado'
    audit_name='R1_C01_2024_POST_FINAL_15Constraints_V01.csv'
    source_bytes=z.read(audit_name)
    assert sha256_bytes(source_bytes)==EXPECTED_CONSTRAINTS_SHA, 'Hash del CSV de restricciones incorrecto'
    manifest=json.loads(z.read('R1_C01_2024_POST_FINAL_Manifest_V01.json'))
    assert manifest['raw_zip_sha256']==EXPECTED_RAW_ZIP_SHA, 'Identidad RAW no coincide'
    assert manifest['status']=='POST_ONLY_NO_FEASIBLE_GLOBAL_SCALE', 'Estado no esperado; auditar versión de origen'
    for fname, digest in manifest.get('outputs_sha256', {}).items():
        assert fname in names and sha256_bytes(z.read(fname))==digest, f'Hash de salida incorrecto: {fname}'
    df=pd.read_csv(io.BytesIO(source_bytes))
    monthly=pd.read_csv(io.BytesIO(z.read('R1_C01_2024_POST_MonthlyAudit_V01.csv')))
    post_csv=pd.read_csv(io.BytesIO(z.read('R1_C01_2024_POST_CANDIDATE_V01.csv')),
                         usecols=['timestamp','GHI_post','DNI_post','DHI_post','cosz_plus','month'])
assert len(post_csv)==17568, 'POST debe tener 17568 filas'
assert manifest['year']==2024 and manifest['post_rows']==17568
assert len(monthly)==12, 'Auditoría mensual incompleta'
assert len(df)==15, 'Se esperan 15 restricciones'
expected={'GHI_annual','DNI_annual','DHI_annual'} | {f'GHI_month_{m:02d}' for m in range(1,13)}
assert set(df.constraint)==expected and df.constraint.is_unique, 'Identidad de las 15 restricciones inválida'
for name in ['E_post','p05','p95','lower','upper']:
    assert np.isfinite(pd.to_numeric(df[name],errors='coerce')).all(), f'No finitos en {name}'
assert (df.E_post>0).all() and (df.p05<=df.p95).all(), 'Energía o límites inconsistentes'
print('PASS insumos, manifest, 15 restricciones y 17568 filas POST')

In [ ]:
# Recalcular INDEPENDIENTEMENTE los intervalos, sin reutilizar lower/upper guardados
TOL_E=1e-6   # kWh/m²: tolerancia de cumplimiento energético, no cambia la factibilidad canónica
TOL_ALPHA=1e-12 # comparación diagnóstica de redondeo para intervalos
calc=df.copy()
calc['lower_recomputed']=calc['p05']/calc['E_post']
calc['upper_recomputed']=calc['p95']/calc['E_post']
calc['lower_residual']=calc['lower']-calc['lower_recomputed']
calc['upper_residual']=calc['upper']-calc['upper_recomputed']
assert calc[['lower_residual','upper_residual']].abs().to_numpy().max()<1e-12, 'Diferencias en intervalos originales'
L=calc.lower_recomputed.max()
U=calc.upper_recomputed.min()
active_lower=calc.loc[np.isclose(calc.lower_recomputed,L,atol=1e-12,rtol=0),'constraint'].tolist()
active_upper=calc.loc[np.isclose(calc.upper_recomputed,U,atol=1e-12,rtol=0),'constraint'].tolist()
alpha_gap=max(0.,L-U)
feasible=bool(L<=U)
print('Límite inferior global:',format(L,'.17g'),'—',active_lower)
print('Límite superior global:',format(U,'.17g'),'—',active_upper)
print('Brecha alpha: ',format(alpha_gap,'.17g'))
print('Factibilidad exacta de los 15 intervalos:',feasible)
assert np.isclose(L,manifest['annual_scale_lower'],atol=1e-12,rtol=0)
assert np.isclose(U,manifest['annual_scale_upper'],atol=1e-12,rtol=0)
assert feasible == bool(manifest['feasible'])
assert not feasible, 'Esta auditoría V01 se elaboró para el caso 2024 no factible; no fabricar FINAL'

In [ ]:
# Contraste de restricciones antagonistas: cada extremo impone un requisito energético específico.
lo=calc.sort_values('lower_recomputed',ascending=False)[['constraint','lower_recomputed','upper_recomputed']].head(5)
hi=calc.sort_values('upper_recomputed',ascending=True)[['constraint','lower_recomputed','upper_recomputed']].head(5)
print('Restricciones con mayores límites inferiores:')
print(lo.to_string(index=False,float_format=lambda x:f'{x:.12f}'))
print('\nRestricciones con menores límites superiores:')
print(hi.to_string(index=False,float_format=lambda x:f'{x:.12f}'))
# Es suficiente un par no solapado para probar incompatibilidad del sistema.
for lower_name in active_lower:
    for upper_name in active_upper:
        lowrow=calc.set_index('constraint').loc[lower_name]
        upro=calc.set_index('constraint').loc[upper_name]
        assert lowrow.lower_recomputed>upro.upper_recomputed
        print(f'CONTRADICCIÓN: {lower_name} requiere α≥{lowrow.lower_recomputed:.12f},'
              f' pero {upper_name} requiere α≤{upro.upper_recomputed:.12f}')

In [ ]:
# Análisis de incumplimientos como DIAGNÓSTICO: tres factores hipotéticos, SIN generar FINAL
# alpha=1 representa POST sin reconciliar; L y U son valores frontera mutuamente incompatibles.
scenarios={'POST_sin_escalar':1.0, 'frontera_inferior_L':float(L),'frontera_superior_U':float(U)}
rows=[]
for scen, alpha in scenarios.items():
    for r in calc.itertuples(index=False):
        E=alpha*r.E_post
        below=max(0.,r.p05-E)
        above=max(0.,E-r.p95)
        rows.append({'scenario':scen,'alpha':alpha,'constraint':r.constraint,
                     'energy_if_scaled_kWh_m2':E,'p05':r.p05,'p95':r.p95,
                     'shortfall_kWh_m2':below,'excess_kWh_m2':above,
                     'violation_kWh_m2':max(below,above),
                     'strict_inside':(below==0 and above==0),
                     'inside_energy_tolerance':max(below,above)<=TOL_E})
scen_df=pd.DataFrame(rows)
print(scen_df.groupby('scenario',sort=False).agg(
    failures_strict=('strict_inside',lambda x:int((~x).sum())),
    failures_TOL_E=('inside_energy_tolerance',lambda x:int((~x).sum())),
    max_violation_kWh_m2=('violation_kWh_m2','max')
).to_string())
# La tolerancia numérica no salva la incompatibilidad 2024.
assert all((scen_df.query('scenario==@scen').violation_kWh_m2>TOL_E).any() for scen in scenarios)

In [ ]:
# Inspección independiente del contenido energético POST: meses y año
post_csv['timestamp']=pd.to_datetime(post_csv['timestamp'])
assert post_csv.timestamp.is_monotonic_increasing and post_csv.timestamp.is_unique
assert post_csv.timestamp.iloc[0]==pd.Timestamp('2024-01-01 00:00:00')
assert post_csv.timestamp.iloc[-1]==pd.Timestamp('2024-12-31 23:30:00')
assert (post_csv.timestamp.diff().iloc[1:]==pd.Timedelta(minutes=30)).all()
assert (post_csv.timestamp.dt.date==pd.Timestamp('2024-02-29').date()).sum()==48, 'Faltan registros bisiestos'
assert (post_csv.month.to_numpy()==post_csv.timestamp.dt.month.to_numpy()).all()
assert np.isfinite(post_csv[['GHI_post','DNI_post','DHI_post','cosz_plus']].to_numpy()).all()
assert (post_csv[['GHI_post','DNI_post','DHI_post']]>=-1e-12).to_numpy().all()
close=(post_csv.GHI_post-post_csv.DNI_post*post_csv.cosz_plus-post_csv.DHI_post).abs()
assert close.max()<1e-6, 'Cierre POST no válido'
monthly_energy=post_csv.groupby('month')[['GHI_post','DNI_post','DHI_post']].sum()*0.5/1000
annual_energy=monthly_energy.sum()
for var in ['GHI','DNI','DHI']:
    label=var+'_annual'
    assert abs(annual_energy[var+'_post']-float(calc.set_index('constraint').loc[label,'E_post']))<1e-6
for m in range(1,13):
    label=f'GHI_month_{m:02d}'
    assert abs(monthly_energy.loc[m,'GHI_post']-float(calc.set_index('constraint').loc[label,'E_post']))<1e-6
rd=monthly_energy['DHI_post']/monthly_energy['GHI_post']
assert np.allclose(rd.to_numpy(),monthly.sort_values('month')['rd_post_E'].to_numpy(),atol=1e-8,rtol=0)
print('PASS recomputación POST desde serie semihoraria. Cierre máximo:',close.max())
print('Energías anuales POST (kWh/m²):',annual_energy.to_dict())

In [ ]:
# Tamaño mínimo de la incompatibilidad para documentación (NO optimización ni corrección)
# En alpha=L, cuantifica el exceso por las restricciones superiores limitantes;
# en alpha=U, cuantifica la deficiencia por las restricciones inferiores limitantes.
active_lo=calc.set_index('constraint').loc[active_lower[0]]
active_hi=calc.set_index('constraint').loc[active_upper[0]]
excess_at_L=float(L*active_hi.E_post-active_hi.p95)
shortfall_at_U=float(active_lo.p05-U*active_lo.E_post)
assert excess_at_L>0 and shortfall_at_U>0
print('En α=L, exceso en',active_upper[0],':',f'{excess_at_L:.9f}','kWh/m²')
print('En α=U, defecto en',active_lower[0],':',f'{shortfall_at_U:.9f}','kWh/m²')
print('Brecha α relativa a U:',f'{alpha_gap/U*100:.6f}%')
# Tolerancia expresada en alfa por restricción, solo para probar distancia de redondeo.
L_tol=np.max((calc.p05-TOL_E)/calc.E_post)
U_tol=np.min((calc.p95+TOL_E)/calc.E_post)
print('Intersección con epsilon energética SOLO diagnóstica:',L_tol,U_tol,'factible:',L_tol<=U_tol)
assert L_tol>U_tol, 'Tolerancia energética altera resultado; revisar fuentes'

In [ ]:
# Exportación de una auditoría SOLO diagnóstica. Nunca exportar un recurso FINAL.
OUTPUT.mkdir(parents=True,exist_ok=False)
files={
    f'{OUTPUT_NAME}_15Constraints.csv':calc,
    f'{OUTPUT_NAME}_ScenarioViolations.csv':scen_df,
    f'{OUTPUT_NAME}_MonthlyPOST_Energy.csv':monthly_energy.reset_index(),
}
for name,data in files.items():
    data.to_csv(OUTPUT/name,index=False,float_format='%.17g')
report={
    'protocol':'R1-C01 V1.1','year':YEAR,
    'stage':'FINAL_FEASIBILITY_DIAGNOSTIC_ONLY',
    'conclusion':'NO_FEASIBLE_COMMON_GLOBAL_SCALAR',
    'input_zip_name':NAME,'input_zip_sha256':sha256_file(SOURCE),
    'constraints_sha256':EXPECTED_CONSTRAINTS_SHA,
    'number_of_constraints':len(calc),'post_rows':len(post_csv),
    'alpha_lower':float(L),'alpha_upper':float(U),
    'alpha_gap':float(alpha_gap),'alpha_gap_relative_pct_of_upper':float(alpha_gap/U*100),
    'active_lower_constraints':active_lower,'active_upper_constraints':active_upper,
    'conflict_excess_at_L_kWh_m2':excess_at_L,
    'conflict_shortfall_at_U_kWh_m2':shortfall_at_U,
    'tolerance_E_kWh_m2':TOL_E,
    'tolerance_diagnostic_alpha':TOL_ALPHA,
    'alpha_interval_with_energy_tolerance':{'lower':float(L_tol),'upper':float(U_tol),'feasible':bool(L_tol<=U_tol)},
    'post_closure_max_Wm2':float(close.max()),
    'final_generated':False,'raw_modified':False,'post_modified':False,
    'input_manifest_status':manifest['status'],
    'software':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}
}
text=[
    '# R1-C01 — Dictamen de factibilidad FINAL 2024 (V01)',
    '', '**Estado:** NO FACTIBLE con un factor global común y las 15 restricciones canónicas.',
    '',f'- Factor mínimo requerido: {L:.15f} ({", ".join(active_lower)}).',
    f'- Factor máximo admitido: {U:.15f} ({", ".join(active_upper)}).',
    f'- Brecha absoluta en alfa: {alpha_gap:.15f}.',
    f'- En α=L, exceso de {active_upper[0]}: {excess_at_L:.9f} kWh/m².',
    f'- En α=U, defecto de {active_lower[0]}: {shortfall_at_U:.9f} kWh/m².',
    f'- Con tolerancia E=10⁻⁶ kWh/m²: sigue sin existir intersección.',
    '', '## Alcance',
    'La inviabilidad es una propiedad de la reconciliación escalar global aplicada a este POST 2024 y estas envolventes 1998–2022. No demuestra inviabilidad de cualquier método de posprocesamiento, ni un error físico de RAW/POST. No se ha generado ni aceptado FINAL.',
    '', 'Las evaluaciones de escenarios son diagnósticas y **no** constituyen recursos candidatos.',
]
(OUTPUT/f'{OUTPUT_NAME}_Dictamen.md').write_text('\n'.join(text)+'\n',encoding='utf-8')
report['outputs_sha256']={f.name:sha256_file(f) for f in sorted(OUTPUT.iterdir()) if f.is_file()}
(OUTPUT/f'{OUTPUT_NAME}_Manifest.json').write_text(json.dumps(report,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
zip_path=BASE/f'{OUTPUT_NAME}.zip'
assert not zip_path.exists(), 'ZIP ya existente, no sobrescribir'
with zipfile.ZipFile(zip_path,'x',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUTPUT.iterdir()): z.write(f,arcname=f.name)
print('PASS diagnóstico; no se generó FINAL')
print('ZIP diagnóstico:',zip_path)
print('SHA256:',sha256_file(zip_path))

## Criterios interpretativos

Una intersección vacía de los 15 intervalos muestra que **no existe una reconciliación por un factor común** que satisfaga las restricciones consideradas, dadas las entradas congeladas. Las distancias a fronteras permiten distinguir este conflicto de una discrepancia numérica. El CSV de escenarios es un análisis contrafactual de restricciones, **no una salida de irradiancia FINAL**.

Próximo paso científico, fuera del alcance de este notebook: comparación estadística de RAW y POST frente a NSRDB 2024 y comparación científica de ambos años 2023–2024.